# 02 — Thermal Breakthrough Analysis

Analyse the spatial and temporal pattern of the cold thermal plume as it 
migrates from the injection wells toward the production wells.

**Prerequisites**: `outputs/Group3.npz` and the config workbook.

No FEFLOW / IFM is required.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

from config import load_config, RESULTS_PATH

cfg  = load_config()
data = np.load(str(RESULTS_PATH.with_suffix('.npz')))
T    = data['T']
times = data['times']

In [ ]:
# Temperature change from initial condition per snapshot
# Slice 2 (reservoir top): nodes [nps, 2*nps]
# We estimate nps from total nodes / n_slices
n_nodes = T.shape[1]
n_slices = 6
nps = n_nodes // n_slices

T_s2 = T[:, nps:2*nps]   # shape (20, nps)
T_ic  = cfg.slice_T[1]    # undisturbed reservoir temperature
dT_s2 = T_s2 - T_ic       # negative = cooling, positive = warming

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
target_indices = [1, 3, 5, 9, -1]   # t ≈ 10, 20, 30, 50, 100 yr

for ax, idx in zip(axes, target_indices):
    ax.hist(dT_s2[idx], bins=50, color='steelblue', alpha=0.7)
    yr = times[idx] / 365.25
    ax.set_title(f't = {yr:.0f} yr')
    ax.set_xlabel('ΔT [°C]')
    ax.axvline(0, color='k', lw=0.8)

axes[0].set_ylabel('Node count')
fig.suptitle('Slice 2 temperature change ΔT = T(t) - T_IC  [Group 3]')
plt.tight_layout()
plt.show()

In [ ]:
# Fraction of reservoir nodes cooled by more than 5 °C
threshold = 5.0   # °C
frac_cooled = (dT_s2 < -threshold).mean(axis=1) * 100.0

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times / 365.25, frac_cooled, 's-', color='steelblue')
ax.set_xlabel('Time [yr]')
ax.set_ylabel(f'Nodes cooled > {threshold} °C  [%]')
ax.set_title(f'Cold plume growth in Slice 2 (ΔT < −{threshold} °C)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()